# 3. Azure AI Search

## 1. What is Azure AI Search?

**Azure AI Search** is Microsoft's managed **enterprise search and retrieval service**.

It can support:

- Full-text/keyword search
- Vector search
- Hybrid search
- Semantic ranking
- Filtering
- Faceting
- Enterprise RAG

For GenAI applications, its most important role is:

> **Retrieve relevant enterprise data and provide it as grounded context to an LLM such as Azure OpenAI.**

---

# 2. Where Does It Fit in RAG?

```text
                Documents
                    │
                    ▼
             Data Ingestion
                    │
                    ▼
             Azure AI Search
                    │
          ┌─────────┼─────────┐
          │         │         │
       Keyword    Vector    Semantic
        Search    Search    Ranking
          │         │         │
          └─────────┼─────────┘
                    ▼
              Top-K Results
                    │
                    ▼
              Azure OpenAI
                    │
                    ▼
              Final Answer
```

Azure AI Search is primarily the **retrieval layer**.

Azure OpenAI is the **generation layer**.

---

# 3. Basic RAG Flow

Suppose you have:

```text
HR Policies
Payroll Documents
Leave Policies
Employee Handbook
```

### Indexing phase

```text
Documents
   ↓
Parse
   ↓
Chunk
   ↓
Generate Embeddings
   ↓
Azure AI Search Index
```

### Query phase

```text
User:
"What is the maternity leave policy?"

        ↓

Azure AI Search
        ↓
Relevant chunks
        ↓
Azure OpenAI
        ↓
Answer
```

---

# 4. What is an Index?

An **index** is the searchable structure that stores your indexed content and associated fields.

Think of it conceptually like a database table optimized for search.

Example:

```text
HRPolicyIndex

┌─────────┬────────────┬───────────────┬─────────────┐
│ id      │ content    │ department    │ embedding   │
├─────────┼────────────┼───────────────┼─────────────┤
│ 101     │ Leave...   │ HR            │ [0.2,...]   │
│ 102     │ Payroll... │ Finance       │ [0.7,...]   │
└─────────┴────────────┴───────────────┴─────────────┘
```

---

# 5. Important Azure AI Search Components

| Component | Purpose |
|---|---|
| **Search Service** | Azure resource providing search capability |
| **Index** | Searchable collection of documents/data |
| **Index Fields** | Defines searchable/filterable/vector data |
| **Indexer** | Automatically pulls data from supported sources and populates an index |
| **Data Source** | Source from which an indexer retrieves data |
| **Skillset** | AI enrichment pipeline during indexing |
| **Search Query** | Retrieves matching documents |
| **Vector Search** | Finds semantically similar vectors |
| **Semantic Ranker** | Improves ranking of search results |

---

# 6. Keyword Search

Traditional search looks for matching words.

User:

```text
"employee vacation policy"
```

The search engine looks for relevant terms such as:

```text
employee
vacation
policy
```

This works well when the query and document contain similar terminology.

---

# 7. Vector Search ⭐⭐⭐⭐⭐

Vector search is critical for RAG.

```text
Document
   ↓
Embedding Model
   ↓
[0.12, 0.43, -0.21, ...]
```

User query:

```text
"How many days can an employee take off?"
```

↓

```text
Query Embedding
       ↓
Vector Search
       ↓
Semantically similar documents
```

It can retrieve relevant content even if the exact words don't match.

For example:

```text
Query:
"How many days can I take off?"

Document:
"Employees are entitled to 20 annual vacation days."
```

The words are different, but their **semantic meaning is related**.

---

# 8. Hybrid Search ⭐⭐⭐⭐⭐

Hybrid search combines:

```text
Keyword Search
       +
Vector Search
       ↓
Combined Results
       ↓
Ranking
```

Example:

```text
User Query
    │
    ├───────────────┐
    ▼               ▼
Keyword Search   Vector Search
    │               │
    └───────┬───────┘
            ▼
      Combined Results
            │
            ▼
      Semantic Ranker
            │
            ▼
         Top-K
```

### Why use hybrid search?

Because keyword and semantic search solve different problems.

**Keyword search** is good for:

- Employee ID
- Product code
- Policy number
- Exact terminology

**Vector search** is good for:

- Semantic similarity
- Natural language questions
- Conceptual queries

Hybrid search gives you both.

---

# 9. Semantic Ranker

After retrieving candidates, **semantic ranking** can improve the ordering of results based on the meaning and relevance of the query and retrieved content.

Conceptually:

```text
Query
 ↓
Search
 ↓
Top 50 candidates
 ↓
Semantic Ranker
 ↓
Top 5 relevant documents
```

This is useful in RAG because sending fewer, higher-quality chunks to the LLM can improve:

- Relevance
- Context quality
- Latency
- Token consumption

---

# 10. Metadata Filtering

Suppose your documents contain:

```text
department = HR
country = India
document_type = policy
year = 2026
```

User asks:

> "What is the leave policy for India?"

You can filter before/alongside retrieval:

```text
country = "India"
AND
department = "HR"
```

Then perform search.

This is especially useful in enterprise RAG.

---

# 11. Azure AI Search + Azure OpenAI

A typical architecture:

```text
                         User
                           │
                           ▼
                    AI Application
                           │
                           ▼
                    User Question
                           │
                           ▼
                  Azure AI Search
                           │
             ┌─────────────┼─────────────┐
             │             │             │
          Keyword       Vector       Semantic
           Search        Search        Ranker
             │             │             │
             └─────────────┼─────────────┘
                           ▼
                     Relevant Chunks
                           │
                           ▼
                    Prompt + Context
                           │
                           ▼
                     Azure OpenAI
                           │
                           ▼
                     Final Answer
```

---

# 12. Indexing Architecture

An enterprise ingestion pipeline can look like:

```text
SharePoint / Blob / SQL
          │
          ▼
      Data Source
          │
          ▼
       Indexer
          │
          ▼
     Skillset / AI
     Enrichment
          │
          ▼
      Azure AI Search
          │
          ▼
        Index
```

Depending on the architecture, you can also build your own ingestion pipeline:

```text
Document
   ↓
Python / LangChain
   ↓
Chunking
   ↓
Embedding
   ↓
Azure AI Search
```

You don't have to use an Azure Search indexer for every RAG implementation.

---

# 13. What is an Indexer?

An **indexer** is an Azure AI Search component that automates the process of pulling data from supported data sources and indexing it.

Conceptually:

```text
Blob Storage
     ↓
   Indexer
     ↓
Azure AI Search Index
```

Instead of writing your own extraction-and-indexing pipeline for every supported source, the indexer can automate much of that ingestion process.

---

# 14. What is a Skillset?

A **skillset** defines AI enrichment operations that can be applied during indexing.

Conceptually:

```text
Document
   ↓
Indexer
   ↓
Skillset
   ├── Extract text
   ├── OCR
   ├── Entity extraction
   └── Other enrichment
   ↓
Search Index
```

This is useful when you want the search pipeline to enrich content before indexing.

---

# 15. Vector Dimension

If your embedding model produces:

```text
1536 dimensions
```

your vector field needs to be configured accordingly.

Conceptually:

```text
Embedding Model
      ↓
1536-dimensional vector
      ↓
Azure AI Search vector field
```

The vector dimensionality needs to be compatible with the embedding model used for indexing and querying.

---

# 16. Azure AI Search vs Qdrant

Since you have Qdrant experience, this is an important comparison.

| Qdrant | Azure AI Search |
|---|---|
| Vector database/search engine | Managed enterprise search service |
| Strong vector search | Vector + keyword + semantic search |
| Open-source | Azure managed service |
| Can self-host | Azure-managed |
| Vector-focused | Broader enterprise search |
| Metadata filtering | Filtering |
| Good for custom RAG | Strong enterprise Azure integration |
| API-based | Azure SDK/REST APIs |
| Less Azure-native | Deep Azure ecosystem integration |

### Interview answer

> "I've worked with Qdrant for vector retrieval. Azure AI Search provides similar vector-search capabilities but is broader because it combines keyword search, vector search, filtering, and semantic ranking, along with tight integration into the Azure ecosystem. For an enterprise Azure RAG solution, I would consider Azure AI Search when the requirements include hybrid search, enterprise data integration, and Azure-native security and operations."

---

# 17. Azure AI Search vs Traditional Database

Don't think of it as simply:

> "Another database."

Its primary purpose is **search and retrieval**.

A traditional database might optimize for:

```text
INSERT
UPDATE
DELETE
TRANSACTION
```

Azure AI Search optimizes for:

```text
SEARCH
RETRIEVAL
RANKING
FILTERING
VECTOR SIMILARITY
SEMANTIC SEARCH
```

---

# 18. RAG Example

User:

> "Can I carry forward unused vacation days?"

Search:

```text
User Query
     ↓
Embedding
     ↓
Azure AI Search
     ↓
Vector + Keyword Search
     ↓
Semantic Ranking
     ↓
Top 5 chunks
```

Prompt:

```text
System:
Answer using only the provided context.

Context:
[Retrieved HR policy chunks]

Question:
Can I carry forward unused vacation days?
```

Then:

```text
Prompt
  ↓
Azure OpenAI
  ↓
Grounded Answer
```

---

# 19. How to Improve Azure AI Search Retrieval

If your RAG accuracy is poor, investigate:

### Data

- Document parsing
- Chunking
- Metadata
- Duplicate content

### Retrieval

- Embedding model
- Vector dimensions
- Top-K
- Similarity
- Hybrid search
- Filters

### Ranking

- Semantic ranker
- Re-ranking

### Generation

- Context construction
- Prompt
- LLM
- Grounding instructions

A good senior-level answer is:

> "I wouldn't immediately change the LLM when RAG accuracy is poor. I would first isolate whether the problem is ingestion, chunking, embedding, retrieval, ranking, context construction, or generation."

---

# 20. Common Interview Questions

### Q1. What is Azure AI Search?

> Azure AI Search is a managed Azure search and retrieval service that supports keyword, vector, hybrid and semantic search capabilities and is commonly used as the retrieval layer for enterprise RAG applications.

### Q2. What is hybrid search?

> Hybrid search combines traditional keyword search with vector-based semantic search to improve retrieval across both exact-match and semantic queries.

### Q3. Why use Azure AI Search for RAG?

> It provides enterprise search capabilities including vector search, hybrid search, filtering, semantic ranking, and Azure-native integration, making it suitable for retrieving grounded context for LLM applications.

### Q4. What is an index?

> An index is the searchable structure containing the fields and content that Azure AI Search uses to retrieve documents.

### Q5. What is an indexer?

> An indexer automates pulling data from supported data sources and populating a search index.

### Q6. What is a skillset?

> A skillset defines AI enrichment operations that can be applied to content during indexing.

### Q7. Vector search vs keyword search?

> Keyword search is primarily based on lexical matching, while vector search uses embeddings to find semantically similar content.

### Q8. Why hybrid search?

> It combines lexical precision with semantic understanding, which is particularly useful for enterprise RAG where queries may contain both exact identifiers and natural-language concepts.

---

# 21. Senior-Level Scenario

### Interviewer:

> "Our RAG system has good LLM performance but poor retrieval accuracy. What would you investigate?"

### Strong answer:

> "I would first separate retrieval quality from generation quality. I would inspect the ingestion and chunking strategy, embedding model and dimensions, metadata, query formulation, top-K, vector similarity, keyword search, and hybrid-search configuration. I would then evaluate semantic ranking or reranking. I would use retrieval metrics such as Recall@K, Precision@K and MRR to quantify the retrieval layer before changing the LLM."

That demonstrates **AI engineering thinking**, rather than simply knowing how to call an LLM.